In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import re 

from sbi.inference import NPE
from sbi.utils import BoxUniform

from tqdm.auto import tqdm

import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path

sims_dir = Path("../COLA_nbody_sims")


## prior

In [ ]:
prior = BoxUniform(
    low=torch.tensor([0.10, 2.7, 0.85]),
    high=torch.tensor([0.50, 3.4, 1.15]),
) # slightly larger than LHS priors

## LHS params


In [ ]:
lhs_params = np.load("bin1_samples.npy")

In [ ]:
def load_boosts(
        boost_dir,
        Bin,
        p=10,
        delta_star=4, 
        R=10
        ):
    
    
    boost_dir = Path(boost_dir)

    pattern = re.compile(
        rf"^boostCOLAz(\d+\.\d{{3}})"
        rf"_p{p}"
        rf"_dstar{delta_star}"
        rf"_R{R}"
        rf"_Bin{Bin}"
        rf"_sim(\d+)\.npz$"
    )
   
    files = sorted(
        f for f in boost_dir.iterdir()
        if pattern.match(f.name)
    )


    if not files:
        raise FileNotFoundError(
            f"no matching boost files found for Bin={Bin}"
            f"p={p}, delta_star={delta_star}, R={R}"
        )

    return [(f, np.load(f)) for f in files]

    

## load boosts

In [ ]:
bin=1

boosts = load_boosts(
    boost_dir="/home/amanih/project_files/boosts",
    Bin=bin, # NOTE: 1 or 5
    delta_star=4, 
    R=10
    )

pk_camb_dir = Path("/home/amanih/project_files/pks_camb")

thetas = []
boost_matter = []
boost_mark = []
boost_cross = []
pks = []

sim_indices = [] # for splitting


# get theta values from lhs params
for filename, data in boosts:
     
     # get sim index
     match = re.search(r"_sim(\d+)\.npz$", filename.name)
     
     if match is None:
          continue
     
     sim_index = int(match.group(1))

     sim_indices.append(sim_index)

     # corresponding lhs params
     theta = lhs_params[sim_index]

     # construct vectors
     thetas.append(theta)
     boost_matter.append(data["boostPk"][1:])
     boost_mark.append(data["boostPkmark"][1:])
     boost_cross.append(data["boostPkcross"][1:])

     z_match = re.search(r"z(\d+\.\d{3})", filename.name)
     redshift = z_match.group(1)

     pk_file = pk_camb_dir / (
         f"Pk_z{redshift}_Bin{bin}_sim{sim_index}.npz"
         )

     if not pk_file.exists():
         raise FileNotFoundError(
             f"No matching CAMB file for sim={sim_index}, z={redshift}"
             )
     
     with np.load(pk_file) as pk_data:
        pk = pk_data["Pk"].copy()
        k = pk_data["k"].copy()

        omega_m_pk = pk_data["omega_m"].item()
        As_pk = pk_data["As"].item()
        
        omega_m_theta = theta[0]
        As_theta = theta[1]
        
        assert np.isclose(omega_m_pk, omega_m_theta)
        assert np.isclose(np.log(1e10 * As_pk), As_theta) # CAMB stores As 
        
        k_boost = data["k"]
        k_camb = pk_data["k"]
        
        # print("Boost:", len(k_boost), k_boost.min(), k_boost.max())
        # print("CAMB:", len(k_camb), k_camb.min(), k_camb.max())

        # print(k_boost)
        # print(k_camb)


        k_boost = np.asarray(data["k"][1:])
        k_camb = np.asarray(k)
        
        # assert np.array_equal(data["k"][1:], k), "k arrays do not match" <-failing test

        pks.append(pk)


assert len(pks) == len(boost_matter)

     

## checking k-grid (ongoing)

In [ ]:
import re

k_groups = {}

for filename, data in boosts:

    name = str(filename)

    match = re.search(
        r"boostCOLAz(\d+\.\d+)_p\d+_dstar\d+_R\d+_Bin\d+_sim(\d+)",
        name
    )

    z = float(match.group(1))
    sim = int(match.group(2))

    kboost = data["k"]

    # find an existing group with an identical k-grid
    found_group = False

    for group_id, group in k_groups.items():
        if np.array_equal(kboost, group["k"]):
            group["members"].append((sim, z))
            found_group = True
            break

    # otherwise create a new group
    if not found_group:
        group_id = len(k_groups)
        k_groups[group_id] = {
            "k": kboost.copy(),
            "members": [(sim, z)]
        }


print(f"total distinct k-grids: {len(k_groups)}\n")

for group_id, group in k_groups.items():
    members = sorted(group["members"], key=lambda x: (x[1], x[0]))

    print(f"Group {group_id}: {len(members)} simulations")
    print([(sim, f"{z:.3f}") for sim, z in members])
    print()

checking all boosts share the same kgrid

In [ ]:
# test - kgrid
k_reference = None

for boost in boosts:
    # with np.load(boost_file) as data:
    #     kboost = data["k"]

    # print(boost[0])
    boost = boost[1]
    kboost=boost['k']

    if k_reference is None:
        k_reference = kboost.copy()
    elif not np.array_equal(kboost, k_reference):
        raise ValueError(f"k grid differs in {boost}")

## split into training and unseen indices

In [ ]:
training_indices = np.arange(1, 150)
train_mask = np.isin(sim_indices, training_indices)
unseen_mask = ~train_mask

# ensure test is not in training sims 

test_sim=180 # redefined later 

sim_indices = np.array(sim_indices)

print("training sims:", np.unique(sim_indices[train_mask]))
print("unseen sims:", np.unique(sim_indices[unseen_mask]))

assert test_sim not in sim_indices[train_mask]
assert test_sim in sim_indices[unseen_mask]

print(f"simulation {test_sim} is held out")

# convert lists (theta, boosts, pk, sim_indices) to arrays
thetas = np.array(thetas)
boost_matter = np.array(boost_matter)
boost_mark = np.array(boost_mark)
boost_cross = np.array(boost_cross)
pks = np.array(pks)
sim_indices = np.array(sim_indices)

# apply masks to split into training/test
theta_train = thetas[train_mask]
boost_matter_train = boost_matter[train_mask]
boost_mark_train = boost_mark[train_mask]
boost_cross_train = boost_cross[train_mask]
pk_train = pks[train_mask]

theta_unseen = thetas[unseen_mask]
boost_matter_unseen = boost_matter[unseen_mask]
boost_mark_unseen = boost_mark[unseen_mask]
boost_cross_unseen = boost_cross[unseen_mask]
pk_unseen = pks[unseen_mask]

In [ ]:
# convert to tensors

theta = torch.tensor(
    np.array(theta_train),
    dtype=torch.float32
)

boost_matter = torch.tensor(
    np.array(boost_matter_train),
    dtype=torch.float32
)

boost_mark = torch.tensor(
    np.array(boost_mark_train),
    dtype=torch.float32
)

boost_cross = torch.tensor(
    np.array(boost_cross_train),
    dtype=torch.float32
)

boost_markcross = torch.cat(
    [boost_mark, boost_cross],
    dim=1
)

pk_camb = torch.tensor(
    np.array(pk_train),
    dtype=torch.float32
)

## train NPE(s)

In [ ]:
def train_posterior(theta, x, prior):
    inference=NPE(prior=prior)
    inference.append_simulations(theta, x)
    density_estimator=inference.train()
    return inference.build_posterior(density_estimator)

In [ ]:
from sklearn.decomposition import PCA

nPCA=15

pca_matter = PCA(n_components=nPCA)
pca_mark = PCA(n_components=nPCA)
pca_cross = PCA(n_components=nPCA)
pca_markcross = PCA(n_components=nPCA)
pca_pk = PCA(n_components=nPCA)


boost_matter_pca = torch.tensor(
    pca_matter.fit_transform(boost_matter.numpy()),
    dtype=torch.float32
)
boost_mark_pca = torch.tensor(
    pca_mark.fit_transform(boost_mark.numpy()),
    dtype=torch.float32
)
boost_cross_pca = torch.tensor(
    pca_cross.fit_transform(boost_cross.numpy()),
    dtype=torch.float32
)

boost_markcross_pca = torch.cat(
    [boost_mark_pca, boost_cross_pca],
    dim=1
)
pk_camb_pca = torch.tensor(
    pca_pk.fit_transform(pk_camb.numpy()),
    dtype=torch.float32
)

In [ ]:
# concatenate PCAs from boosts, Pk CAMB

# Pk + mm (matter auto)
pk_bmm_pca = torch.cat(
    [pk_camb_pca, boost_matter_pca],
    dim=1
)

# Pk + MM (marked auto)
pk_bMM_pca = torch.cat(
    [pk_camb_pca, boost_mark_pca],
    dim=1
)

# Pk + mM (cross-spectrum)
pk_bMm_pca = torch.cat(
    [pk_camb_pca, boost_cross_pca],
    dim=1
)

# Pk + MM + mM (marked stats)
pk_bMM_bMm_pca = torch.cat(
    [pk_camb_pca, boost_mark_pca, boost_cross_pca],
    dim=1
)

# Pk + Bmm + BMm + BMM
pk_all_pca = torch.cat(
    [
        pk_camb_pca,
        boost_matter_pca,
        boost_mark_pca,
        boost_cross_pca
    ],
    dim=1
)

In [ ]:
# train 

posterior_matter=train_posterior(theta, boost_matter_pca, prior) 
posterior_mark=train_posterior(theta, boost_mark_pca, prior) 
posterior_cross=train_posterior(theta, boost_cross_pca, prior) 
posterior_markcross=train_posterior(theta, boost_markcross_pca, prior) 
posterior_pk=train_posterior(theta, pk_camb_pca, prior) 

posterior_pk_bmm = train_posterior(
    theta, pk_bmm_pca, prior
)

posterior_pk_bMM = train_posterior(
    theta, pk_bMM_pca, prior
)

posterior_pk_bMm = train_posterior(
    theta, pk_bMm_pca, prior
)


posterior_pk_bMM_bMm = train_posterior(
    theta, pk_bMM_bMm_pca, prior
)

posterior_pk_all = train_posterior(
    theta, pk_all_pca, prior
)

In [ ]:
# set test simulation (sim_index, z) -- for now i manually change this
test_sim = 180
test_z = 0.000

# load boost
boost_file = Path(
    f"/home/amanih/project_files/boosts/"
    f"boostCOLAz{test_z:.3f}_p10_dstar4_R10_Bin{bin}_sim{test_sim}.npz"
)

with np.load(boost_file) as data:
    test_boost_matter = data["boostPk"][1:]
    test_boost_mark = data["boostPkmark"][1:]
    test_boost_cross = data["boostPkcross"][1:]

# load CAMB Pk
pk_test = pk_camb_dir / (
    f"Pk_z{test_z:.3f}_Bin{bin}_sim{test_sim}.npz"
)

with np.load(pk_test) as data:
    pk_test = data["Pk"].copy()

# true value
theta_true = torch.tensor(
    lhs_params[test_sim],
    dtype=torch.float32
)

print(theta_true)

In [ ]:
# convert test to tensor

boost_matter_obs = torch.tensor(
    test_boost_matter,
    dtype=torch.float32
)

boost_mark_obs = torch.tensor(
    test_boost_mark,
    dtype=torch.float32
)

boost_cross_obs = torch.tensor(
    test_boost_cross,
    dtype=torch.float32
)

boost_markcross_obs = torch.cat(
    [boost_mark_obs, boost_cross_obs],
    dim=0
)

pk_obs = torch.tensor(
    pk_test,
    dtype=torch.float32
)

In [ ]:
# fit PCA to test

boost_matter_obs_pca = torch.tensor(
    pca_matter.transform(
        boost_matter_obs.unsqueeze(0).numpy()
    ).squeeze(0),
    dtype=torch.float32
)

boost_mark_obs_pca = torch.tensor(
    pca_mark.transform(
        boost_mark_obs.unsqueeze(0).numpy()
    ).squeeze(0),
    dtype=torch.float32
)

boost_cross_obs_pca = torch.tensor(
    pca_cross.transform(
        boost_cross_obs.unsqueeze(0).numpy()
    ).squeeze(0),
    dtype=torch.float32
)

boost_markcross_obs_pca = torch.cat(
    [boost_mark_obs_pca, boost_cross_obs_pca],
    dim=0
)

pk_obs_pca = torch.tensor(
    pca_pk.transform(
        pk_obs.unsqueeze(0).numpy()
    ).squeeze(0),
    dtype=torch.float32
)

In [ ]:
# Pk + mm (matter auto)
pk_bmm_obs_pca = torch.cat(
    [pk_obs_pca, boost_matter_obs_pca],
    dim=0
)

# Pk + MM (marked auto)
pk_bMM_obs_pca = torch.cat(
    [pk_obs_pca, boost_mark_obs_pca],
    dim=0
)

# Pk + mM (cross-spectrum)
pk_bMm_obs_pca = torch.cat(
    [pk_obs_pca, boost_cross_obs_pca],
    dim=0
)

# Pk + MM + mM (marked stats)
pk_bMM_bMm_obs_pca = torch.cat(
    [pk_obs_pca, boost_mark_obs_pca, boost_cross_obs_pca]
)

# Pk + Bmm + BMm + BMM
pk_all_obs_pca = torch.cat(
    [
        pk_obs_pca,
        boost_matter_obs_pca,
        boost_mark_obs_pca,
        boost_cross_obs_pca
    ],
    dim=0
)

In [ ]:
samples_matter = posterior_matter.sample(
    (5000,),
    x=boost_matter_obs_pca
)

samples_mark = posterior_mark.sample(
    (5000,),
    x=boost_mark_obs_pca
)

samples_cross = posterior_cross.sample(
    (5000,),
    x=boost_cross_obs_pca
)

samples_markcross = posterior_markcross.sample(
    (5000,),
    x=boost_markcross_obs_pca
)

samples_pk = posterior_pk.sample(
    (5000,),
    x=pk_obs_pca
)

In [ ]:
samples_pk_bmm = posterior_pk_bmm.sample(
    (5000,),
    x=pk_bmm_obs_pca
)

samples_pk_bMM = posterior_pk_bMM.sample(
    (5000,),
    x=pk_bMM_obs_pca
)

samples_pk_bMm = posterior_pk_bMm.sample(
    (5000,),
    x=pk_bMm_obs_pca
)

samples_pk_bMM_bMm = posterior_pk_bMM_bMm.sample(
    (5000,),
    x=pk_bMM_bMm_obs_pca
)

samples_pk_all = posterior_pk_all.sample(
    (5000,),
    x=pk_all_obs_pca
)

## save posteriors

In [ ]:
# save posteriors 

torch.save(
    {
        "samples": samples_matter,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10}A_s$"
        ],
    },
    f"posteriors/corner_matter_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

torch.save(
    {
        "samples": samples_mark,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10}A_s$"
        ],
    },
    f"posteriors/corner_mark_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

torch.save(
    {
        "samples": samples_cross,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10}A_s$"
        ],
    },
    f"posteriors/corner_cross_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

torch.save(
    {
        "samples": samples_markcross,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10}A_s$"
        ],
    },
    f"posteriors/corner_markcross_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

torch.save(
    {
        "samples": samples_pk,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10}A_s$"
        ],
    },
    f"posteriors/corner_pk_CAMB_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

# Pk + Bmm
torch.save(
    {
        "samples": samples_pk_bmm,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10} A_s$"
        ],
    },
    f"posteriors/corner_pk_bmm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

# Pk + BMM
torch.save(
    {
        "samples": samples_pk_bMM,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10} A_s$"
        ],
    },
    f"posteriors/corner_pk_bMM_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

# Pk + BMm
torch.save(
    {
        "samples": samples_pk_bMm,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10} A_s$"
        ],
    },
    f"posteriors/corner_pk_bMm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

# Pk + BMM + BMm
torch.save(
    {
        "samples": samples_pk_bMM_bMm,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10} A_s$"
        ],
    },
    f"posteriors/corner_pk_bMM_bMm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

# Pk + Bmm + BMm + BMM
torch.save(
    {
        "samples": samples_pk_all,
        "truth": theta_true,
        "labels": [
            r"$\Omega_m$",
            r"$\ln(10^{10} A_s$"
        ],
    },
    f"posteriors/corner_pk_all_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt"
)

## corner plots

In [ ]:
from getdist import MCSamples, plots

In [ ]:
matter = torch.load(f'posteriors/corner_matter_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
mark  = torch.load(f'posteriors/corner_mark_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
cross = torch.load(f'posteriors/corner_cross_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
markcross = torch.load(f'posteriors/corner_markcross_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
pk = torch.load(f'posteriors/corner_pk_CAMB_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')

mcsamples = []
truth_values = matter['truth']

for pca in [matter, mark, cross, markcross, pk]:
    samples = pca['samples'].numpy().astype(np.float64)
    labels = [
        r"\Omega_m",
        r"\ln(10^{10} A_s)",
        r"\mu"
    ]
    mcsamples.append(
        MCSamples(
            samples=samples,
            labels=labels,
            names=["Om",  "As", "mu"]
        )
    )

cmap = plt.get_cmap("prism") 
colors = [cmap(i) for i in np.linspace(0,1,len(mcsamples))]


g=plots.get_subplot_plotter()
g.triangle_plot(
    mcsamples, 
    filled=True,
    markers=truth_values,
    legend_labels=[f"{type}" for type in ['matter auto', 'marked auto', 'cross', 'mark + cross', 'pk']],
    contour_colors=colors,
    contour_ls = ['--', ':', '-', '-'],

)

plt.suptitle(f"SBI Bin 1 - Boosts and CAMB P(k)", y=1.04, fontsize=15)
plt.subplots_adjust(top=0.95)

plt.savefig("SBI_COLA_boost_bin1.png", dpi=150, bbox_inches="tight")

plt.show()

In [ ]:

pk_bmm = torch.load(f'posteriors/corner_pk_bmm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
pk_bMM = torch.load(f'posteriors/corner_pk_bMM_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
pk_bMm = torch.load(f'posteriors/corner_pk_bMm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
pk_bMM_bMm = torch.load(f'posteriors/corner_pk_bMM_bMm_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')
pk_all = torch.load(f'posteriors/corner_pk_all_COLA_Bin{bin}_sim{test_sim}_z{test_z}_train{len(training_indices)}.pt')

In [ ]:

mcsamples = []
truth_values = pk_bMM['truth']

for posterior in [pk_bMm, pk_bMM_bMm, pk_bmm, pk_bMM, pk_all]:
    samples = posterior['samples'].numpy().astype(np.float64)
    labels = [r'\Omega_m', r'A_s', r'\mu']

    mcsamples.append(
        MCSamples(samples=samples, labels=labels, names=["Om", "As", "mu"])
    )

colors = [ 'lightseagreen', 'orange', 'magenta', 'blue', 'red']

g = plots.get_subplot_plotter()
g.triangle_plot(
    mcsamples,
    filled=True,
    markers=truth_values,
    legend_labels=[
        'Pk + marked auto + cross',
        'Pk + cross',
        'Pk + matter auto',
        'Pk + marked auto',
        'Pk + all'
    ],
    contour_colors=colors
)

plt.suptitle(
    "SBI Bin 1: Combinations of CAMB P(k) with various Boost stats",
    y=1.04,
    fontsize=15
)
plt.subplots_adjust(top=0.95)
plt.savefig("SBI_COLA_pk_combinations_bin1.png", dpi=150, bbox_inches="tight")
plt.show()

## PCA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

X = boost_matter_pca.numpy()
theta_np = theta.numpy()

df = pd.DataFrame({
    "Omega_m": theta_np[:, 0],
    "ln10As": theta_np[:, 1],
    "mu": theta_np[:, 2],
})

for i in range(15):
    df[f"PC{i+1}"] = X[:, i]

g = sns.PairGrid(
    df,
    x_vars=["Omega_m", "ln10As", "mu"],
    y_vars=[f"PC{i+1}" for i in range(15)],
    height=2.5
)

g.map(sns.scatterplot, s=15, alpha=0.4, color='blue')

plt.suptitle("PCA Mode Dependence on LHS params - matter boost")
plt.subplots_adjust(top=0.96)

plt.show()

In [ ]:
X = boost_mark_pca.numpy()
theta_np = theta.numpy()

df = pd.DataFrame({
    "Omega_m": theta_np[:, 0],
    "ln10As": theta_np[:, 1],
    "mu": theta_np[:, 2],
})

for i in range(15):
    df[f"PC{i+1}"] = X[:, i]

g = sns.PairGrid(
    df,
    x_vars=["Omega_m", "ln10As", "mu"],
    y_vars=[f"PC{i+1}" for i in range(15)],
    height=2.5
)

g.map(sns.scatterplot, s=15, alpha=0.4, color='magenta')

plt.suptitle("PCA Mode Dependence on LHS params - mark boost")
plt.subplots_adjust(top=0.96)

plt.show()

In [ ]:
X = boost_cross_pca.numpy()
theta_np = theta.numpy()

df = pd.DataFrame({
    "Omega_m": theta_np[:, 0],
    "ln10As": theta_np[:, 1],
    "mu": theta_np[:, 2],
})

for i in range(15):
    df[f"PC{i+1}"] = X[:, i]

g = sns.PairGrid(
    df,
    x_vars=["Omega_m", "ln10As", "mu"],
    y_vars=[f"PC{i+1}" for i in range(15)],
    height=2.5
)

g.map(sns.scatterplot, s=15, alpha=0.4, color='orange')

plt.suptitle("PCA Mode Dependence on LHS params - cross boost")
plt.subplots_adjust(top=0.96)

plt.show()

In [ ]:
X = pk_camb_pca.numpy()
theta_np = theta.numpy()

df = pd.DataFrame({
    "Omega_m": theta_np[:, 0],
    "ln10As": theta_np[:, 1],
    "mu": theta_np[:, 2],
})

for i in range(nPCA):
    df[f"PC{i+1}"] = X[:, i]

g = sns.PairGrid(
    df,
    x_vars=["Omega_m", "ln10As", "mu"],
    y_vars=[f"PC{i+1}" for i in range(nPCA)],
    height=2.5
)

g.map(sns.scatterplot, s=15, alpha=0.4, color = 'green')

plt.suptitle("PCA Mode Dependence on LHS params - CAMB Pk")
plt.subplots_adjust(top=0.96)

plt.show()